# Lab Iceberg Avancé — Jour 1

**Profil docker :** `iceberg` · **Durée :** 90 min · **Format :** Trio rôlé (Driver / Reviewer / Client)

---

## Objectifs
1. Inspecter les fichiers de metadata Iceberg dans MinIO
2. Comparer Copy-on-Write (CoW) et Merge-on-Read (MoR) avec benchmarks
3. Configurer et mesurer l'impact de la compaction
4. Appliquer le hidden partitioning et l'évolution de partition
5. Répondre à la **question client** finale

---

## Contexte — Incident RetailCo

> La table `transactions` de RetailCo dégrade ses performances de 15 % par semaine.
> Les requêtes Trino sur `store_region` mettent 45 secondes alors que le SLA est 10 secondes.
> Votre mission : diagnostiquer, corriger, benchmarker.

---

## Stack
- **Spark 3.5** avec Iceberg 1.5 (catalog Polaris + Nessie)
- **Trino 435** pour les requêtes analytiques
- **MinIO** (s3://warehouse/) comme stockage

> **Formateur :** Les cellules `# TODO` sont à compléter. Les cellules `# ✅ SOLUTION` sont masquées avant distribution.

---
## Setup — Connexion à la stack

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyiceberg.catalog.rest import RestCatalog
import requests, json, boto3, time, os

# ── SparkSession avec Iceberg + Polaris + Nessie ──────────────────────────────
spark = (
    SparkSession.builder
    .appName('RetailCo-Iceberg-Lab-J1')
    .config('spark.sql.extensions', 'org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,org.projectnessie.spark.extensions.NessieSparkSessionExtensions')
    .getOrCreate()
)

print(f'✅ Spark : {spark.version}')
print(f'✅ Catalog Polaris : polaris')
print(f'✅ Catalog Nessie  : nessie (branche main)')

# ── Clients ───────────────────────────────────────────────────────────────────
MINIO    = 'http://minio:9000'
POLARIS  = 'http://polaris:8181/api/catalog'
NESSIE   = 'http://nessie:19120/api/v2'
WAREHOUSE = 's3a://warehouse/'

s3 = boto3.client('s3', endpoint_url=MINIO,
    aws_access_key_id='minioadmin', aws_secret_access_key='minioadmin')

print('\n📦 Buckets MinIO disponibles:')
for b in s3.list_buckets()['Buckets']:
    print(f'   s3://{b["Name"]}')

---
## ⏱️ Temps 1 — Kata d'amorce (10 min)

Créez une table Iceberg RetailCo dans Polaris, observez les fichiers générés dans MinIO.

In [ ]:
# ── Création de la table Bronze RetailCo ─────────────────────────────────────
spark.sql("CREATE NAMESPACE IF NOT EXISTS polaris.retailco")

spark.sql("""
    CREATE TABLE IF NOT EXISTS polaris.retailco.transactions (
        transaction_id  STRING,
        store_id        STRING,
        store_region    STRING,
        category        STRING,
        unit_price      DOUBLE,
        quantity        INT,
        total_amount    DOUBLE,
        return_flag     BOOLEAN,
        transaction_ts  TIMESTAMP
    )
    USING iceberg
    PARTITIONED BY (store_region, days(transaction_ts))
    TBLPROPERTIES (
        'write.format.default' = 'parquet',
        'write.parquet.compression-codec' = 'snappy',
        'format-version' = '2'
    )
""")
print('✅ Table polaris.retailco.transactions créée')

# Charger le dataset sample
df = spark.read.csv('s3a://bronze/retailco/transactions/retailco_transactions_sample.csv',
                    header=True, inferSchema=True)
df.write.format('iceberg').mode('append') \
  .saveAsTable('polaris.retailco.transactions')

print(f'✅ {df.count()} lignes chargées')

# ── Observer la structure dans MinIO ──────────────────────────────────────────
print('\n📂 Fichiers dans s3://warehouse/retailco/transactions/')
try:
    objs = s3.list_objects_v2(Bucket='warehouse', Prefix='retailco/transactions/')
    for obj in objs.get('Contents', []):
        size = obj['Size']
        key = obj['Key']
        print(f'   {size:>8} bytes  {key}')
except Exception as e:
    print(f'   Bucket pas encore peuplé: {e}')

---
## ⏱️ Temps 2 — Incident à résoudre (50 min)

### 📋 Partie A — Inspection des internals Iceberg (15 min)

> Objectif : comprendre ce que contient chaque fichier de metadata avant de toucher à quoi que ce soit.

In [ ]:
# ── Inspection du snapshot actuel ────────────────────────────────────────────
print('📋 Snapshots de la table :')
spark.sql('SELECT snapshot_id, committed_at, operation, summary FROM polaris.retailco.transactions.snapshots') \
     .show(truncate=False)

print('\n📋 Manifests du dernier snapshot :')
spark.sql('SELECT path, length, added_data_files_count, added_rows_count FROM polaris.retailco.transactions.manifests') \
     .show(truncate=False)

print('\n📋 Fichiers de données :')
spark.sql('SELECT file_path, file_format, record_count, file_size_in_bytes FROM polaris.retailco.transactions.files') \
     .show(truncate=False)

In [ ]:
# ── Lire un fichier metadata.json directement ────────────────────────────────
# C'est ce que fait Iceberg à chaque lecture de table

print('📁 Fichiers metadata dans MinIO :')
try:
    resp = s3.list_objects_v2(Bucket='warehouse',
                              Prefix='retailco/transactions/metadata/')
    meta_files = [obj['Key'] for obj in resp.get('Contents', [])]
    for f in meta_files:
        print(f'   {f}')

    # Lire le dernier fichier metadata.json
    if meta_files:
        latest = sorted([f for f in meta_files if f.endswith('.json')])[-1]
        content = s3.get_object(Bucket='warehouse', Key=latest)['Body'].read()
        meta = json.loads(content)
        print(f'\n📄 Contenu de {latest.split("/")[-1]} :')
        print(f'   format-version   : {meta.get("format-version")}')
        print(f'   table-uuid       : {meta.get("table-uuid")}')
        print(f'   current-snapshot : {meta.get("current-snapshot-id")}')
        print(f'   schemas          : {len(meta.get("schemas", []))} version(s)')
        print(f'   partition-specs  : {len(meta.get("partition-specs", []))} spec(s)')
        print(f'   snapshots        : {len(meta.get("snapshots", []))} snapshot(s)')
except Exception as e:
    print(f'Erreur: {e}')

### 📋 Partie B — CoW vs MoR : l'incident de performance (20 min)

> **Contexte** : RetailCo fait des DELETE pour le RGPD (droit à l'effacement).
> Avec CoW (défaut), chaque DELETE réécrit tout le fichier.
> Avec MoR, le DELETE écrit juste un fichier de positional deletes.
>
> **Problème** : la table accumule des positional delete files non compactés → lecture lente.

**TODO 1** : Comparer le temps d'un DELETE sur une table CoW vs MoR sur les mêmes données.

In [ ]:
# ── Table CoW (Copy-on-Write) — défaut ───────────────────────────────────────
spark.sql("""
    CREATE TABLE IF NOT EXISTS polaris.retailco.transactions_cow
    USING iceberg
    TBLPROPERTIES (
        'write.delete.mode' = 'copy-on-write',
        'write.update.mode' = 'copy-on-write',
        'write.merge.mode'  = 'copy-on-write',
        'format-version'    = '2'
    )
    AS SELECT * FROM polaris.retailco.transactions
""")

# ── Table MoR (Merge-on-Read) ─────────────────────────────────────────────────
spark.sql("""
    CREATE TABLE IF NOT EXISTS polaris.retailco.transactions_mor
    USING iceberg
    TBLPROPERTIES (
        'write.delete.mode' = 'merge-on-read',
        'write.update.mode' = 'merge-on-read',
        'write.merge.mode'  = 'merge-on-read',
        'format-version'    = '2'
    )
    AS SELECT * FROM polaris.retailco.transactions
""")

print('✅ Tables CoW et MoR créées')

In [ ]:
# TODO 1 — Benchmarker DELETE sur CoW vs MoR
# 📝 Supprimez les 5 transactions avec le plus haut total_amount
#    sur les deux tables. Mesurez le temps d'exécution.
#    Puis comptez les fichiers dans s3://warehouse/ pour chaque table.

# TODO : compléter les benchmarks
print('💡 Indice : time.time() avant et après spark.sql("DELETE FROM ...")')
print('💡 Indice : spark.sql("SELECT * FROM table.files").count() pour compter les fichiers')

In [ ]:
# ✅ SOLUTION TODO 1 — masquer avant distribution

for table, label in [('transactions_cow','CoW'), ('transactions_mor','MoR')]:
    t0 = time.time()
    spark.sql(f"""
        DELETE FROM polaris.retailco.{table}
        WHERE transaction_id IN (
            SELECT transaction_id FROM polaris.retailco.{table}
            ORDER BY total_amount DESC LIMIT 5
        )
    """)
    elapsed = round(time.time() - t0, 2)
    n_files = spark.sql(f'SELECT count(*) AS n FROM polaris.retailco.{table}.files').collect()[0]['n']
    print(f'{label:5} → DELETE en {elapsed}s · {n_files} fichiers de données')

print('\n💡 Observation :')
print('  CoW : DELETE rapide sur petite table mais réécrit TOUT le fichier')
print('  MoR : DELETE écrit un petit fichier de positional deletes — plus rapide à l\'écriture')
print('  MoR : mais la LECTURE doit fusionner data files + delete files — coût à la lecture')

### 📋 Partie C — Compaction : résoudre la dégradation (15 min)

> **L'incident** : la table MoR a accumulé 47 delete files sur 3 mois.
> Chaque lecture scanne et fusionne ces fichiers → latence ×5.

**TODO 2** : Déclencher la compaction et mesurer le gain de performance.

In [ ]:
# Simulation : générer beaucoup de petits fichiers et delete files
print('⏳ Simulation de 30 micro-batchs (accumulation de petits fichiers)...')
for i in range(30):
    spark.sql(f"""
        INSERT INTO polaris.retailco.transactions_mor
        SELECT transaction_id || '-v{i}', store_id, store_region, category,
               unit_price * 1.01, quantity, total_amount * 1.01,
               return_flag, transaction_ts
        FROM polaris.retailco.transactions LIMIT 3
    """)
print('✅ 30 micro-batchs insérés — petits fichiers accumulés')

before_files = spark.sql('SELECT count(*) AS n FROM polaris.retailco.transactions_mor.files').collect()[0]['n']
print(f'   Nombre de fichiers avant compaction : {before_files}')

In [ ]:
# TODO 2 — Compaction
# 📝 a) Mesurez le temps d'une requête COUNT avant compaction
#    b) Déclenchez la compaction avec spark.sql('CALL ...')
#    c) Mesurez le temps de la même requête après compaction
#    d) Comptez les fichiers avant / après

print('💡 Procédure Iceberg : CALL polaris.system.rewrite_data_files(table => "retailco.transactions_mor")')
print('💡 Options : strategy => "binpack", target-file-size-bytes => 134217728')

In [ ]:
# ✅ SOLUTION TODO 2

# Avant compaction
t0 = time.time()
cnt = spark.sql('SELECT count(*) FROM polaris.retailco.transactions_mor').collect()[0][0]
elapsed_before = round(time.time() - t0, 3)
print(f'Avant compaction : {cnt} lignes en {elapsed_before}s')

# Compaction
print('\n⏳ Compaction en cours (bin-packing, target 128 Mo)...')
t0 = time.time()
spark.sql("""
    CALL polaris.system.rewrite_data_files(
        table => 'retailco.transactions_mor',
        strategy => 'binpack',
        options => map('target-file-size-bytes', '134217728', 'min-input-files', '2')
    )
""")
elapsed_compact = round(time.time() - t0, 1)
print(f'✅ Compaction terminée en {elapsed_compact}s')

after_files = spark.sql('SELECT count(*) AS n FROM polaris.retailco.transactions_mor.files').collect()[0]['n']

# Après compaction
t0 = time.time()
cnt = spark.sql('SELECT count(*) FROM polaris.retailco.transactions_mor').collect()[0][0]
elapsed_after = round(time.time() - t0, 3)

print(f'\nAprès compaction  : {cnt} lignes en {elapsed_after}s')
print(f'Réduction fichiers : {before_files} → {after_files}')
print(f'Gain de performance : {round(elapsed_before/max(elapsed_after,0.001), 1)}×')

### 📋 Partie D — Hidden Partitioning (10 min)

> La table est partitionnée par `store_region` et `days(transaction_ts)`.
> Mais les requêtes des analystes filtrent souvent sur `month(transaction_ts)`.
> On veut migrer vers `month()` sans réécrire toutes les données.

**TODO 3** : Ajouter une spec de partition `months(transaction_ts)` et observer l'évolution.

In [ ]:
# TODO 3 — Evolution de partition
# 📝 Ajoutez la spec month(transaction_ts) à la table existante
#    Sans réécrire les données existantes
#    Observez que les deux specs coexistent dans les metadata

print('💡 ALTER TABLE polaris.retailco.transactions REPLACE PARTITION FIELD ...')
print('💡 Inspecter : SELECT * FROM polaris.retailco.transactions.partitions')

In [ ]:
# ✅ SOLUTION TODO 3

# Remplacer days() par months() — les anciennes partitions restent accessibles
spark.sql("""
    ALTER TABLE polaris.retailco.transactions
    REPLACE PARTITION FIELD days(transaction_ts)
    WITH months(transaction_ts)
""")
print('✅ Partition evolution : days → months')
print('📋 Specs de partition actives :')
spark.sql('SELECT * FROM polaris.retailco.transactions.partitions LIMIT 10').show(truncate=False)
print('\n💡 Les anciennes données partitionnées par JOUR restent lisibles.')
print('   Les nouvelles insertions seront partitionnées par MOIS.')
print('   Iceberg gère les deux specs simultanément — zero réécriture.')

---
## ⏱️ Temps 3 — Extension (20 min) · Pour les profils avancés

**Objectif** : Sort order + Z-ordering pour accélérer les requêtes sur plusieurs colonnes.

In [ ]:
# ── Sort Order + Rewrite avec Z-ordering ─────────────────────────────────────
# Mesurer l'impact sur une requête filtrant sur store_region ET category

query = "SELECT sum(total_amount) FROM polaris.retailco.transactions WHERE store_region = 'Île-de-France' AND category = 'Électronique'"

t0 = time.time()
spark.sql(query).collect()
print(f'Avant Z-order : {round(time.time()-t0, 3)}s')

# Appliquer le sort order
spark.sql("""
    CALL polaris.system.rewrite_data_files(
        table => 'retailco.transactions',
        strategy => 'sort',
        sort_order => 'zorder(store_region, category)'
    )
""")

t0 = time.time()
spark.sql(query).collect()
print(f'Après Z-order  : {round(time.time()-t0, 3)}s')
print('\n💡 Le Z-ordering co-localise les données multi-colonnes — réduit les scans.')

---
## ⏱️ Temps 4 — Débrief client (10 min)

### Question obligatoire — rôle Client :

1. **"Pourquoi vous avez choisi MoR plutôt que CoW pour cette table ?"**
2. **"Si je fais 500 suppressions RGPD par jour, quand dois-je déclencher la compaction ?"**
3. **"Le changement de partitionnement de jours vers mois — ça a un impact sur mes requêtes en cours ?"**
4. **"Quel est le coût de la compaction en termes de compute ? Doit-on la prévoir en dehors des heures de pic ?"**

---

## Récapitulatif

| Concept | Ce que vous avez fait | Emporté |
|---------|----------------------|---------|
| Metadata inspection | Lire manifest files + snapshots dans MinIO | Comprendre ce qu'Iceberg écrit vraiment |
| CoW vs MoR | Benchmark DELETE sur les deux modes | Choisir selon write vs read amplification |
| Compaction | Déclencher + mesurer le gain | Planifier la compaction en prod |
| Partition evolution | Migrer de days → months sans réécriture | Faire évoluer le partitionnement sans downtime |
| Z-ordering | Sort order multi-colonnes | Accélérer les requêtes filtrées sur N colonnes |

In [ ]:
# Nettoyage optionnel (ne pas exécuter si vous voulez garder les tables pour J2)
print('Pour nettoyer : spark.sql("DROP TABLE IF EXISTS polaris.retailco.transactions_cow")')
print('               spark.sql("DROP TABLE IF EXISTS polaris.retailco.transactions_mor")')